# Verify quantization and autocast gates

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thehalleyyoung/tensorguard/blob/main/examples/tutorials/08_quantization.ipynb)

Quantized and mixed-precision deployments fail when calibration, boundary, dtype, or backend assumptions drift. TensorGuard exposes small gates that can run before conversion or export.

In [ ]:
%pip install -q "git+https://github.com/thehalleyyoung/tensorguard.git"

In [ ]:
import torch, torch.nn as nn
import torch.ao.quantization as tq
from tensorguard import verify_mixed_precision, verify_quantization_eager

class QuantReady(nn.Module):
    def __init__(self):
        super().__init__()
        self.quant = tq.QuantStub()
        self.fc = nn.Linear(4, 3)
        self.dequant = tq.DeQuantStub()
    def forward(self, x):
        return self.dequant(self.fc(self.quant(x)))

model = QuantReady().eval()
model.qconfig = tq.default_qconfig
prepared = tq.prepare(model, inplace=False)
uncalibrated = verify_quantization_eager(prepared)
print('uncalibrated quantization OK:', uncalibrated.ok)
assert not uncalibrated.ok
with torch.no_grad():
    prepared(torch.randn(2, 4))
calibrated = verify_quantization_eager(prepared)
print('calibrated quantization OK:', calibrated.ok)
assert calibrated.ok

The same deployment pass can check autocast policy choices:

In [ ]:
mp = verify_mixed_precision(nn.Linear(4, 3).eval(),
                            backend='cpu', autocast_dtype=torch.bfloat16)
print('mixed precision OK:', mp.ok)
assert mp.ok